In [2]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset
from datasets import load_dataset

from transformers import (
    set_seed,
)

from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from unsloth import FastLanguageModel
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
from huggingface_hub import hf_hub_download

def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)

/tmp/ipykernel_50427/1161939413.py:17: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 10-18 05:27:21 [__init__.py:244] Automatically detected platform cuda.


In [21]:
dataset_2 = load_dataset("anonymousOWSHateLLM/Hate.2_labels_labeled")
df_2_label = pd.DataFrame(dataset_2["2_labels"])


README.md: 0.00B [00:00, ?B/s]

data/2_labels-00000-of-00001.parquet:   0%|          | 0.00/43.5M [00:00<?, ?B/s]

Generating 2_labels split:   0%|          | 0/240647 [00:00<?, ? examples/s]

In [4]:
print("Total size of 2 labels data: ",df_2_label.shape[0])
print("Language counts ",df_2_label['language'].value_counts())

Total size of 2 labels data:  240647
Language counts  language
deu    125617
eng    108375
vie      6655
Name: count, dtype: int64


In [5]:
mstral7b = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
gemma9b = "unsloth/gemma-2-9b-it-bnb-4bit"
qwen14b = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
llama8B = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"



# Labeling for 2 labels: Hate  / Neutral. LightGBM, Vote, Mean Average

In [6]:
columns = [qwen14b, llama8B, mstral7b, gemma9b]

LightGbm

In [17]:


with open('lgb_model_label_1.pkl', 'rb') as f:
    lgb_label_1 = pickle.load(f)

c1 = [x + "_label_1" for x in columns]
X = df_2_label[c1]
val_preds = lgb_label_1.predict(X, num_iteration=lgb_label_1.best_iteration)
df_2_label["lgb_label_1"] = val_preds



with open('lgb_model_label_2.pkl', 'rb') as f:
    lgb_label_2 = pickle.load(f)

c2 = [x + "_label_2" for x in columns]
X = df_2_label[c2]
val_preds = lgb_label_2.predict(X, num_iteration=lgb_label_2.best_iteration)
df_2_label["lgb_label_2"] = val_preds



index_1 = df_2_label['lgb_label_1'] > df_2_label['lgb_label_2']
index_2 = df_2_label['lgb_label_1'] > .0

index_all = index_1 & index_2

df_2_label['Lgb'] = 2
df_2_label.loc[index_all, 'Lgb'] = 1
df_2_label['Lgb'].value_counts()

Lgb
2    237525
1      3122
Name: count, dtype: int64

Voting

In [11]:
index_qwen = df_2_label[qwen14b + "_label_1"] > .5
index_llama8b = df_2_label[llama8B+ "_label_1"] > .5
index_gemma9b = df_2_label[gemma9b+ "_label_1"] > .5
index_mstral7b = df_2_label[mstral7b+ "_label_1"] > .5

from collections import Counter

index_all = index_mstral7b.astype(int) + index_gemma9b.astype(int) + index_llama8b.astype(int) + index_qwen.astype(int) 
index_all = index_all >= 2

df_2_label['Vote'] = 2
df_2_label.loc[index_all, 'Vote'] = 1
df_2_label['Vote'].value_counts()

Vote
2    235940
1      4707
Name: count, dtype: int64

Mean Average

In [12]:
c1 = [x + "_label_1" for x in columns]
df_2_label["mean_label_1"] = df_2_label[c1].mean(axis=1)

c2 = [x + "_label_2" for x in columns]
df_2_label["mean_label_2"] = df_2_label[c2].mean(axis=1)

index_1 = df_2_label['mean_label_1'] > df_2_label['mean_label_2']
index_2 = df_2_label['mean_label_1'] > .0
index_all = index_1 & index_2

df_2_label['Mean'] = 2
df_2_label.loc[index_all, 'Mean'] = 1
df_2_label['Mean'].value_counts()

Mean
2    236660
1      3987
Name: count, dtype: int64